# Publish raw passed-bill data

This notebook filters enacted bills, inspects one Knesset request, and publishes the raw bill-detail API responses to a public Hugging Face dataset repository.

Published dataset: https://huggingface.co/datasets/nickbes/lawsofisrael

# 1. Filter passed bills with links

In [ ]:
import json
import time
from pathlib import Path

import pandas as pd
import requests
from huggingface_hub import HfApi, notebook_login
from openpyxl import load_workbook
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

DATA_PATH = next(Path("../dataset").glob("*.xlsx"))
PASSED_STATUS = "התקבלה בקריאה שלישית"

raw = pd.read_excel(DATA_PATH)
sheet = load_workbook(DATA_PATH, data_only=False, read_only=True).active
link_column = next(cell.column for cell in sheet[1] if cell.value == "קישור")
link_formulas = pd.Series(
    [row[link_column - 1].value for row in sheet.iter_rows(min_row=2)],
    index=raw.index,
    dtype="string",
)
page_urls = link_formulas.str.extract(r'(https?://[^"]+)', expand=False)
bill_ids = page_urls.str.extract(r'/bills/(\d+)', expand=False)

passed_bills = (
    raw.assign(page_url=page_urls, bill_id=bill_ids)
    .loc[
        lambda data: data["סטטוס"].eq(PASSED_STATUS) & data["bill_id"].notna(),
        ["bill_id", "מספר כנסת", "סוג הצעת חוק", "שם", "סטטוס", "page_url"],
    ]
    .rename(
        columns={
            "מספר כנסת": "knesset_num",
            "סוג הצעת חוק": "bill_type",
            "שם": "source_name",
            "סטטוס": "source_status",
        }
    )
)
passed_bills["bill_id"] = passed_bills["bill_id"].astype("int64")

print(f"Passed bills with links: {len(passed_bills):,}")
print(passed_bills["bill_type"].value_counts().to_string())
passed_bills.head()

# 2. Inspect one GET response

A direct scripted GET to the human-facing page may return the Knesset anti-bot shell (HTTP 247). The page itself loads its bill data from the public `GetLegislationBillItem` JSON endpoint, so both responses are printed below.

In [ ]:
DETAIL_API = (
    "https://www.knesset.gov.il/WebSiteApi/knessetapi/"
    "LegislationItem/GetLegislationBillItem"
)
HEADERS = {"User-Agent": "lawsofisrael-data-collection/0.1"}

retry = Retry(
    total=4,
    backoff_factor=0.5,
    status_forcelist=(429, 500, 502, 503, 504),
    allowed_methods={"GET"},
)
http = requests.Session()
http.headers.update(HEADERS)
http.mount("https://", HTTPAdapter(max_retries=retry))

sample_bill = passed_bills.loc[passed_bills["bill_type"].eq("פרטית")].iloc[0]
page_response = http.get(sample_bill["page_url"], timeout=30)
print("Page URL:", page_response.url)
print("Page status:", page_response.status_code)
print("Page response preview:")
print(page_response.text[:500])

api_response = http.get(
    DETAIL_API, params={"ItemId": int(sample_bill["bill_id"])}, timeout=30
)
api_response.raise_for_status()
sample_payload = api_response.json()
print("\nData URL:", api_response.url)
print("Data status:", api_response.status_code)
print("Data response preview:")
print(api_response.text[:2000])

# 3. Scrape raw JSON once

This one-time collector requests each passed bill, keeps the unmodified API responses in memory, and writes no local dataset or cache files.

In [ ]:
REQUEST_DELAY_SECONDS = 0.2
payload_by_id = {}
failures = []

for position, bill_id in enumerate(passed_bills["bill_id"], start=1):
    try:
        response = http.get(
            DETAIL_API, params={"ItemId": int(bill_id)}, timeout=30
        )
        response.raise_for_status()
        payload_by_id[bill_id] = response.json()
    except (requests.RequestException, ValueError) as error:
        failures.append({"bill_id": int(bill_id), "error": str(error)})
    if position % 25 == 0:
        print(f"Collected {len(payload_by_id):,} responses")
    time.sleep(REQUEST_DELAY_SECONDS)

if failures:
    raise RuntimeError(f"Failed requests: {failures}")

raw_payloads = [payload_by_id[bill_id] for bill_id in passed_bills["bill_id"]]
expected_ids = set(passed_bills["bill_id"])
payload_ids = [payload["general"]["Id"] for payload in raw_payloads]
if len(payload_ids) != len(set(payload_ids)) or set(payload_ids) != expected_ids:
    raise ValueError("Raw payload IDs do not match the filtered passed bills")

print(f"Collected {len(raw_payloads):,} raw responses in memory")

## Raw JSON structure

The published snapshot is a JSON list with **573 objects**, one object per passed bill. Values are mostly Hebrew, many fields are nullable or empty, and dates appear as both ISO timestamps and display-formatted strings.

```json
{
  "general": {"Id": 0, "Name": "...", "Status": "...", "Initiators": "..."},
  "correction": {"OriginalOf": [], "FixItems": [], "CancelOf": []},
  "sessionAndDocs": {
    "Sessions": [{"StepTitle": "...", "StartDate": "...", "ProtocolUrl": "..."}],
    "DraftLaws": [{"FileText": "...", "FilePath": "...", "FileDate": null}],
    "LegalDocuments": [{"FileText": "...", "FilePath": "...", "FileDate": null}]
  },
  "unionMerge": {},
  "actions": [],
  "potentials": [],
  "secondaryLawInstalled": [],
  "secondaryLawInProcess": []
}
```

- **`general`**: bill ID and title, Knesset number, proposal type, status, private-bill number, committee, commencement date, publication references, summary, previous names, initiator names, and joiners. `Initiators` is a comma-separated string and is populated for 199 private bills in this snapshot; party affiliation is **not** supplied by this endpoint.
- **`sessionAndDocs.Sessions`**: the legislative timeline, including step, date, location, committee/plenum identifiers, protocol URL, broadcast URL, and vote identifiers.
- **Document collections**: `DraftLaws`, `LegalDocuments`, `GovernmentDocuments`, and background/accompanying-document lists. Entries use `FileText`, `FilePath`, and `FileDate`; `FilePath` points to the available PDF, DOCX, or external document.
- **`correction`**: laws this bill originates from, directly or indirectly amends, cancels, or is affected by.
- **`unionMerge`**: continuity, split, and merged/leading-bill relationships.
- **Other arrays**: `actions`, `potentials`, `secondaryLawInstalled`, and `secondaryLawInProcess` contain optional process or secondary-legislation relationships and are often empty.

### Where is the final file?

- **Final enacted law**: inspect `sessionAndDocs.LegalDocuments` and select the item whose `FileText` is `חוק - פרסום ברשומות`. Its `FilePath` is the official published-law PDF. It is present for **572 of 573** bills in this snapshot. `חוק - נוסח לא רשמי`, when present, is an unofficial alternative rather than the authoritative publication.
- **Final bill before enactment**: inspect `sessionAndDocs.DraftLaws` and select an item whose `FileText` contains `לקריאה השנייה והשלישית`. This includes corrected or re-laid variants and is present for all **573** bills.

```python
legal_documents = bill["sessionAndDocs"]["LegalDocuments"]
official_law = next(
    (document for document in legal_documents
     if document.get("FileText") == "חוק - פרסום ברשומות"),
    None,
)
official_law_url = (
    official_law["FilePath"].replace("\\", "/")
    if official_law else None
)

draft_documents = bill["sessionAndDocs"]["DraftLaws"]
final_bill = next(
    (document for document in draft_documents
     if "לקריאה השנייה והשלישית" in document.get("FileText", "")),
    None,
)
final_bill_url = (
    final_bill["FilePath"].replace("\\", "/")
    if final_bill else None
)
```

This is the unmodified API response shape. Any later semantic labels, normalized people/party tables, or simplified civilian summaries should be stored as derived data rather than overwriting these raw records.

# 4. Log in to Hugging Face

Run this cell with a write-capable personal account. Authentication is stored by the Hugging Face SDK outside the notebook; no token is embedded in the notebook.

In [ ]:
notebook_login()

# 5. Create and publish the dataset

The repository is public for transparency. Raw JSON and the provenance card are uploaded directly from memory, then verified on the Hub.

In [ ]:
DATASET_NAME = "lawsofisrael"
DATASET_PATH = "data/passed_bills_raw.json"

api = HfApi()
namespace = api.whoami()["name"]
repo_id = f"{namespace}/{DATASET_NAME}"
repo_url = api.create_repo(
    repo_id=repo_id,
    repo_type="dataset",
    private=False,
    exist_ok=True,
)
api.update_repo_settings(repo_id, repo_type="dataset", private=False)

dataset_bytes = (
    json.dumps(raw_payloads, ensure_ascii=False, indent=2) + "\n"
).encode("utf-8")
api.upload_file(
    path_or_fileobj=dataset_bytes,
    path_in_repo=DATASET_PATH,
    repo_id=repo_id,
    repo_type="dataset",
    commit_message="Publish raw passed-bill responses",
)

dataset_card = f"""---
language:
- he
pretty_name: Passed Israeli Bills - Raw Knesset API
tags:
- law
- israel
- knesset
---

# Passed Israeli Bills - Raw Knesset API

Raw responses from the Knesset bill-detail API for {len(raw_payloads):,} bills whose source status is `{PASSED_STATUS}`.

## Contents

`{DATASET_PATH}` is a UTF-8 JSON list. Records are preserved as returned by the API without semantic classification or enrichment.

## Provenance

- Source: https://main.knesset.gov.il/Activity/Legislation/Laws/Pages/LawBill.aspx
- Detail API: `{DETAIL_API}`
- Collection code: https://github.com/nickBes/lawsofisrael/blob/main/notebooks/scrape_passed_bills.ipynb
- Filter: passed in the third reading and containing a valid Knesset bill link
"""
api.upload_file(
    path_or_fileobj=dataset_card.encode("utf-8"),
    path_in_repo="README.md",
    repo_id=repo_id,
    repo_type="dataset",
    commit_message="Add dataset provenance card",
)

required_files = {DATASET_PATH, "README.md"}
remote_files = set(api.list_repo_files(repo_id, repo_type="dataset"))
missing_files = required_files - remote_files
if missing_files:
    raise RuntimeError(f"Upload verification failed; missing: {sorted(missing_files)}")

print(f"Published {len(raw_payloads):,} records to {repo_url}")